In [1]:
# Function 4 - BBO Capstone Project
# Strategy: Gaussian Process + EI/UCB Hybrid
# Function 4 is a 4D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 4 data
# --------------------------------------------------

X = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.88935640, 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.00625040, 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.28707610],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.75709150, 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.62607060, 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.21196510, 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.85653480],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.90277010, 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.53123150],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.47236690, 0.45319200, 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.85481080, 0.49396462, 0.73530997, 0.80809201],

    # Iteration results from our project
    [0.94838900, 0.89451300, 0.85163700, 0.55219600],
    [0.36207700, 0.85906700, 0.99657800, 0.04075300],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.42433200, 0.45025600, 0.30289300, 0.39324200],
    [0.40870900, 0.31386400, 0.92182800, 0.24917400],
    [0.36255700, 0.53002600, 0.13751200, 0.79845200],
    [0.69535300, 0.59183500, 0.32580700, 0.80070600],
    [0.25715600, 0.02995400, 0.13420300, 0.49166000],
    [0.19058000, 0.24752100, 0.40924200, 0.77761900],
    [0.15283900, 0.19396800, 0.10221200, 0.91813300],
    [0.17897900, 0.24053200, 0.40588000, 0.77214500],
    [0.75077200, 0.29182200, 0.55654600, 0.51163500],
])

y = np.array([
    -22.10828779,
    -14.60139663,
    -11.69993246,
    -16.05376511,
    -10.06963343,
    -15.48708254,
    -12.68168498,
    -16.02639977,
    -17.04923465,
    -12.74176599,
    -27.31639636,
    -13.52764887,
    -16.67911520,
    -16.50715856,
    -17.81799934,
    -26.56182083,
    -12.75832422,
    -19.44155762,
    -28.90327367,
    -13.70274694,
    -29.42709140,
    -11.56574199,
    -26.85778644,
    -7.96677535,
    -6.70208925,
    -32.62566022,
    -19.98949793,
    -4.02554228,
    -13.12278233,
    -23.13942840,

    # Iteration outputs from our project
    32.6256313116837,
    -29.3180112390671,
    -4.02554024114528,
    -1.09887429667435,
    -15.441692398348177,
    -12.747445556701674,
    -14.892779030902826,
    -12.236831492946127,
    -11.35195593241809,
    -20.768342260299686,
    -11.467365274332213,
    -11.037100076992534,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format a point for BBO portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep all values inside the portal domain [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates, model, y_best, xi=0.01):
    """Expected Improvement acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates, model, kappa=2.0):
    """Upper Confidence Bound acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and current best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Gaussian Process model
# --------------------------------------------------
# Matern kernel is suitable for nonlinear, uneven black-box functions.
# Function 4 is 4D, so the model uses four length scales.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(1e-3, 1e3),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    alpha=1e-8,
    n_restarts_optimizer=20,
    random_state=42
)

gpr.fit(X, y)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training score:", round(gpr.score(X, y), 4))


# --------------------------------------------------
# 5. Generate candidate points
# --------------------------------------------------
# Strategy:
# 50% local exploitation around the best point
# 50% global exploration using Latin Hypercube.
# Function 4 is 4D, so we keep more exploration than Function 1 or 2.

np.random.seed(42)

local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.10,
    size=(5000, 4)
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=4, seed=42)
global_candidates = sampler.random(n=5000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])


# --------------------------------------------------
# 6. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates=X_candidates,
    model=gpr,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates=X_candidates,
    model=gpr,
    kappa=UCB_KAPPA
)

# Normalize scores before combining
ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid score:
# EI supports exploitation, UCB supports exploration.
hybrid_score = 0.60 * ei_norm + 0.40 * ucb_norm


# --------------------------------------------------
# 7. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

predicted_y, predicted_std = gpr.predict(
    x_next.reshape(1, -1),
    return_std=True
)

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y[0])
print("Uncertainty:", predicted_std[0])
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 8. Show top 5 candidate points
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]
    mean_i, std_i = gpr.predict(point.reshape(1, -1), return_std=True)

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={mean_i[0]:.6f}, "
        f"std={std_i[0]:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )

Shape of X: (42, 4)
Shape of y: (42,)

Best point so far:
0.948389-0.894513-0.851637-0.552196
Best output so far: 32.6256313116837


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.01. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP Model Diagnostics:
Kernel: 0.757**2 * Matern(length_scale=[148, 0.222, 0.287, 0.227], nu=2.5) + WhiteKernel(noise_level=0.01)
Training score: 0.5267

Next Point to Sample:
X = [0.75736785 0.27923717 0.00789794 0.00363031]
Submission format: 0.757368-0.279237-0.007898-0.003630
Predicted y: -13.054885963540361
Uncertainty: 7.5961407027049335
Hybrid score: 0.9320054625632532

Top 5 Candidates:
1. X=0.757368-0.279237-0.007898-0.003630 | pred=-13.054886, std=7.596141, score=0.932005
2. X=0.221957-0.311018-0.007422-0.105264 | pred=-11.630820, std=7.361747, score=0.931849
3. X=0.086377-0.318427-0.003860-0.141192 | pred=-11.087487, std=7.268883, score=0.921073
4. X=0.878460-0.318109-0.029293-0.052475 | pred=-12.113973, std=7.418027, score=0.863130
5. X=0.259486-0.278742-0.021489-0.071876 | pred=-12.142255, std=7.420267, score=0.856438
